In [1]:
import numpy as np
import os
from tensorflow.keras.models import load_model
from sklearn.metrics import classification_report
from tqdm import tqdm

In [2]:
# Load the pre-trained model
model = load_model('3DCNN_Kiwi__VIS_PCA_model.h5')

# Load PCA components and mean
pca_components = np.load(r'C:\Users\rafin\Desktop\HSIC\Ripeness\Datasets\CNN_Datasets\Kiwi_VIS\PCA\PCA_comp\Kiwi_VIS_pca_comp.npy')
pca_mean = np.load(r'C:\Users\rafin\Desktop\HSIC\Ripeness\Datasets\CNN_Datasets\Kiwi_VIS\PCA\PCA_comp\Kiwi_VIS_pca_mean.npy')

input_directory = 'D:/CSE499A/Datasets/Ripeness/v1/Kiwi_Extract_V1/Ripeness_npy/VIS_resized_Split/test'

In [3]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv3d (Conv3D)             (None, 30, 30, 140, 32)   896       
                                                                 
 average_pooling3d (AverageP  (None, 15, 15, 70, 32)   0         
 ooling3D)                                                       
                                                                 
 conv3d_1 (Conv3D)           (None, 13, 13, 68, 64)    55360     
                                                                 
 conv3d_2 (Conv3D)           (None, 11, 11, 66, 128)   221312    
                                                                 
 average_pooling3d_1 (Averag  (None, 5, 5, 33, 128)    0         
 ePooling3D)                                                     
                                                                 
 flatten (Flatten)           (None, 105600)            0

In [7]:
# Collect all true labels and predicted labels
true_labels = []
predicted_labels = []

# Iterate over each class directory
for class_label in tqdm(os.listdir(input_directory), desc='Processing classes'):
    
    class_dir = os.path.join(input_directory, class_label)
    
    if not os.path.isdir(class_dir):
        continue
    
    # Process each image in the class directory
    for file in tqdm(os.listdir(class_dir), desc=f'Processing {class_label}'):
        
        if not file.endswith('.npy'):
            continue
        
        file_path = os.path.join(class_dir, file)
        image = np.load(file_path)  # Shape: (250, 150, 252)
        
        # Generate mask to exclude background (assuming sum of bands > 0)
        sum_bands = np.sum(image, axis=2)
        mask = sum_bands > 0  # Adjust threshold if necessary
        
        height, width, _ = image.shape
        cube_size = 32
        max_h = height - cube_size
        max_w = width - cube_size
        
        cubes = []
        attempts = 0
        max_attempts = 1000
        
        # Collect 10 valid cubes
        while len(cubes) < 20 and attempts < max_attempts:
            
            h_start = np.random.randint(0, max_h + 1)
            w_start = np.random.randint(0, max_w + 1)
            
            # Check if the entire cube is within non-masked area
            if mask[h_start:h_start+cube_size, w_start:w_start+cube_size].all():
                
                cube = image[h_start:h_start+cube_size, w_start:w_start+cube_size, :]
                cubes.append(cube)
                
            attempts += 1
        
        if len(cubes) < 10:
            
            print(f"Warning: Only found {len(cubes)} cubes for {file}. Skipping.")
            
            continue
        
        # Process each cube and predict
        pred_classes = []
        
        for cube in cubes:
            
            # Apply PCA
            cube_flat = cube.reshape(-1, 224)  
            cube_centered = cube_flat - pca_mean
            cube_pca = np.dot(cube_centered, pca_components.T) 
            cube_pca = cube_pca.reshape(32, 32, 142)
            
            # Z-score normalization per band
            mean = np.mean(cube_pca, axis=(0, 1))
            std = np.std(cube_pca, axis=(0, 1))
            std[std == 0] = 1e-6  # Avoid division by zero
            cube_normalized = (cube_pca - mean) / std
            
            # Prepare input for the model (add channel and batch dimensions)
            cube_input = np.expand_dims(cube_normalized, axis=-1)  # (32, 32, 23, 1)
            cube_input = np.expand_dims(cube_input, axis=0)        # (1, 32, 32, 23, 1)
            
            # Predict
            pred = model.predict(cube_input, verbose=0)
            pred_class = np.argmax(pred, axis=1)[0]
            pred_classes.append(pred_class)
        
        # Majority voting
        majority_vote = np.bincount(pred_classes).argmax()
        predicted_labels.append(majority_vote)
        true_labels.append(class_label)

Processing classes: 100%|██████████| 3/3 [00:29<00:00,  9.89s/it]


In [8]:
# Generate classification report
# Modify this list to match your actual class names in order
CLASS_NAMES = ['Overripe', 'Perfect', 'Unripe']

# Then use this for mapping instead of sorted(unique_labels)
label_to_index = {label: idx for idx, label in enumerate(CLASS_NAMES)}

#unique_labels = sorted(np.unique(true_labels))
#label_to_index = {label: idx for idx, label in enumerate(unique_labels)}

# Convert string labels to indices using predefined CLASS_NAMES
true_indices = [label_to_index[label] for label in true_labels]

# Get all possible class indices
all_labels = list(range(len(CLASS_NAMES)))

print(classification_report(
    true_indices, 
    predicted_labels, 
    labels=all_labels,
    target_names=CLASS_NAMES,
    zero_division=0,
    digits=4
))

              precision    recall  f1-score   support

    Overripe     0.0000    0.0000    0.0000         6
     Perfect     0.4167    1.0000    0.5882        10
      Unripe     0.0000    0.0000    0.0000         8

    accuracy                         0.4167        24
   macro avg     0.1389    0.3333    0.1961        24
weighted avg     0.1736    0.4167    0.2451        24

